In [1]:
from pathlib import Path
import pandas as pd

project_folder = Path(r"D:\electricity-tariff-optimisation")
processed_folder = project_folder / "data" / "processed"

band_file = processed_folder / "customer_design_band_kwh_2013.csv"
feature_file = processed_folder / "customer_design_features_2013.csv"

design_band_kwh = pd.read_csv(
    band_file,
    index_col="household_id",
)

customer_features = pd.read_csv(
    feature_file,
    index_col="household_id",
)

bands = ["Peak", "Shoulder", "Off-peak"]
design_band_kwh = design_band_kwh[bands]

assert design_band_kwh.index.is_unique
assert customer_features.index.is_unique
assert len(design_band_kwh) == 248
assert set(design_band_kwh.index) == set(customer_features.index)
assert design_band_kwh.notna().all().all()
assert (design_band_kwh >= 0).all().all()

print("Design-period data loaded.")
print(f"Households: {len(design_band_kwh)}")
print(f"Total energy (kWh): {design_band_kwh.sum().sum():,.3f}")
print("Validation-period data has not been loaded.")

display(design_band_kwh.head())

Design-period data loaded.
Households: 248
Total energy (kWh): 549,198.974
Validation-period data has not been loaded.


,Peak,Shoulder,Off-peak
household_id,,,
MAC000002,331.363000,1470.438001,385.727000
MAC000040,780.067999,3655.637001,1240.951001
MAC000100,376.286000,1523.439999,224.158000
MAC000145,366.632999,1534.940000,437.267000
MAC000162,170.857000,701.104000,222.892000


In [2]:
from itertools import product

baseline_rate = 0.14228

variable_cost_rates = pd.Series({
    "Peak": 0.15,
    "Shoulder": 0.10,
    "Off-peak": 0.06,
})

minimum_margin_pct = 25.0
maximum_bill_increase_pct = 2.0

price_grid = {
    "Peak": [0.18, 0.20, 0.22, 0.24],
    "Shoulder": [0.12, 0.13, 0.14, 0.15],
    "Off-peak": [0.06, 0.08, 0.10],
}

price_combinations = list(product(
    price_grid["Peak"],
    price_grid["Shoulder"],
    price_grid["Off-peak"],
))

baseline_charges = (
    design_band_kwh.sum(axis=1) * baseline_rate
)

customer_variable_costs = (
    design_band_kwh
    .mul(variable_cost_rates, axis="columns")
    .sum(axis=1)
)

print(f"Price combinations: {len(price_combinations)}")
print(f"Minimum contribution margin (%): {minimum_margin_pct:.1f}")
print(f"Maximum household bill increase (%): {maximum_bill_increase_pct:.1f}")
print(f"Baseline energy charges (GBP): {baseline_charges.sum():,.2f}")
print(f"Assumed variable costs (GBP): {customer_variable_costs.sum():,.2f}")
print("Standing charges and behavioural changes are excluded.")
print("Scenario assumes all selected households use the candidate tariff.")

Price combinations: 48
Minimum contribution margin (%): 25.0
Maximum household bill increase (%): 2.0
Baseline energy charges (GBP): 78,140.03
Assumed variable costs (GBP): 54,672.64
Standing charges and behavioural changes are excluded.
Scenario assumes all selected households use the candidate tariff.


In [3]:
search_results = []

total_variable_cost = customer_variable_costs.sum()
total_baseline_charge = baseline_charges.sum()

for peak_rate, shoulder_rate, off_peak_rate in price_combinations:
    rates = pd.Series({
        "Peak": peak_rate,
        "Shoulder": shoulder_rate,
        "Off-peak": off_peak_rate,
    })

    candidate_charges = (
        design_band_kwh
        .mul(rates, axis="columns")
        .sum(axis=1)
    )

    revenue = candidate_charges.sum()
    contribution = revenue - total_variable_cost
    margin_pct = contribution / revenue * 100

    bill_increase_pct = (
        (candidate_charges / baseline_charges - 1) * 100
    )

    margin_pass = margin_pct >= minimum_margin_pct - 1e-9
    bill_pass = (
        bill_increase_pct.max()
        <= maximum_bill_increase_pct + 1e-9
    )
    rate_order_pass = peak_rate > shoulder_rate > off_peak_rate

    search_results.append({
        "peak_rate": peak_rate,
        "shoulder_rate": shoulder_rate,
        "off_peak_rate": off_peak_rate,
        "revenue_gbp": revenue,
        "contribution_margin_pct": margin_pct,
        "total_customer_saving_gbp": total_baseline_charge - revenue,
        "max_bill_increase_pct": bill_increase_pct.max(),
        "households_paying_more": int(
            (bill_increase_pct > 1e-9).sum()
        ),
        "households_above_increase_limit": int(
            (
                bill_increase_pct
                > maximum_bill_increase_pct + 1e-9
            ).sum()
        ),
        "margin_constraint_pass": margin_pass,
        "bill_constraint_pass": bill_pass,
        "feasible": margin_pass and bill_pass and rate_order_pass,
    })

tariff_search = pd.DataFrame(search_results)

feasible_tariffs = (
    tariff_search.loc[tariff_search["feasible"]]
    .sort_values("revenue_gbp")
    .reset_index(drop=True)
)

tariff_search.to_csv(
    processed_folder / "design_tariff_grid_search_2013.csv",
    index=False,
)

print(f"Price combinations evaluated: {len(tariff_search)}")
print(
    "Combinations meeting the margin constraint:",
    int(tariff_search["margin_constraint_pass"].sum()),
)
print(
    "Combinations meeting the bill constraint:",
    int(tariff_search["bill_constraint_pass"].sum()),
)
print(f"Feasible combinations: {len(feasible_tariffs)}")

if feasible_tariffs.empty:
    print("No tested combination meets all constraints.")
    print("Constraints have not been changed.")
else:
    display(feasible_tariffs.head().round(4))

Price combinations evaluated: 48
Combinations meeting the margin constraint: 30
Combinations meeting the bill constraint: 6
Feasible combinations: 0
No tested combination meets all constraints.
Constraints have not been changed.


In [4]:
comparison_columns = [
    "peak_rate",
    "shoulder_rate",
    "off_peak_rate",
    "contribution_margin_pct",
    "total_customer_saving_gbp",
    "max_bill_increase_pct",
    "households_above_increase_limit",
]

margin_compliant = (
    tariff_search.loc[
        tariff_search["margin_constraint_pass"]
    ]
    .sort_values(
        ["max_bill_increase_pct", "revenue_gbp"]
    )
)

bill_compliant = (
    tariff_search.loc[
        tariff_search["bill_constraint_pass"]
    ]
    .sort_values(
        "contribution_margin_pct",
        ascending=False,
    )
)

print("Lowest household increases among margin-compliant tariffs:")
display(
    margin_compliant[comparison_columns]
    .head(3)
    .round(4)
)

print("Highest margins among bill-compliant tariffs:")
display(
    bill_compliant[comparison_columns]
    .head(3)
    .round(4)
)

print("Diagnostic comparison only; no tariff has been selected.")
print("Original constraints remain unchanged.")

Lowest household increases among margin-compliant tariffs:


,peak_rate,shoulder_rate,off_peak_rate,contribution_margin_pct,total_customer_saving_gbp,max_bill_increase_pct,households_above_increase_limit
7,0.18,0.14,0.08,25.6072,4648.2279,5.2032,8
8,0.18,0.14,0.10,27.8499,2363.7663,6.3109,15
17,0.20,0.13,0.10,26.1375,4120.5275,7.4328,8


Highest margins among bill-compliant tariffs:


,peak_rate,shoulder_rate,off_peak_rate,contribution_margin_pct,total_customer_saving_gbp,max_bill_increase_pct,households_above_increase_limit
4,0.18,0.13,0.08,21.9033,8133.6543,1.2608,0
2,0.18,0.12,0.10,20.5402,9334.6191,-1.5739,0
3,0.18,0.13,0.06,19.2689,10418.1158,0.1532,0


Diagnostic comparison only; no tariff has been selected.
Original constraints remain unchanged.


In [5]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
from scipy.optimize import linprog

energy_matrix = design_band_kwh[bands].to_numpy()
portfolio_energy = energy_matrix.sum(axis=0)

required_revenue = (
    total_variable_cost
    / (1 - minimum_margin_pct / 100)
)

household_charge_limits = (
    baseline_charges.to_numpy()
    * (1 + maximum_bill_increase_pct / 100)
)

constraint_matrix = np.vstack([
    energy_matrix,
    -portfolio_energy.reshape(1, -1),
])

constraint_limits = np.concatenate([
    household_charge_limits,
    [-required_revenue],
])

rate_bounds = [
    (min(price_grid[band]), max(price_grid[band]))
    for band in bands
]

optimisation_result = linprog(
    c=portfolio_energy,
    A_ub=constraint_matrix,
    b_ub=constraint_limits,
    bounds=rate_bounds,
    method="highs",
)

print(f"Solver status: {optimisation_result.message}")

if optimisation_result.success:
    continuous_rates = pd.Series(
        optimisation_result.x,
        index=bands,
        name="rate_gbp_per_kwh",
    )

    continuous_charges = (
        design_band_kwh
        .mul(continuous_rates, axis="columns")
        .sum(axis=1)
    )

    continuous_revenue = continuous_charges.sum()
    continuous_margin_pct = (
        (continuous_revenue - total_variable_cost)
        / continuous_revenue * 100
    )

    continuous_bill_increase_pct = (
        continuous_charges / baseline_charges - 1
    ) * 100

    display(continuous_rates.to_frame().round(8))

    print(f"Contribution margin (%): {continuous_margin_pct:.6f}")
    print(
        "Maximum household bill increase (%):",
        f"{continuous_bill_increase_pct.max():.6f}",
    )
    print(
        "Total customer saving (GBP):",
        f"{total_baseline_charge - continuous_revenue:,.2f}",
    )
    print("Results use unrounded rates.")
elif optimisation_result.status == 2:
    print("No feasible tariff exists within the original rate bounds.")
    print("Original constraints remain unchanged.")
else:
    print("The solver did not establish feasibility or infeasibility.")

Solver status: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)
No feasible tariff exists within the original rate bounds.
Original constraints remain unchanged.


In [7]:
margin_limit_result = linprog(
    c=-portfolio_energy,
    A_ub=energy_matrix,
    b_ub=household_charge_limits,
    bounds=rate_bounds,
    method="highs",
)

if margin_limit_result.success:
    diagnostic_rates = pd.Series(
        margin_limit_result.x,
        index=bands,
        name="diagnostic_rate_gbp_per_kwh",
    )

    diagnostic_charges = (
        design_band_kwh
        .mul(diagnostic_rates, axis="columns")
        .sum(axis=1)
    )

    maximum_revenue = diagnostic_charges.sum()

    maximum_margin_pct = (
        (maximum_revenue - total_variable_cost)
        / maximum_revenue * 100
    )

    diagnostic_bill_increase_pct = (
        diagnostic_charges / baseline_charges - 1
    ) * 100

    display(diagnostic_rates.to_frame().round(8))

    print(
        "Maximum achievable contribution margin (%):",
        f"{maximum_margin_pct:.4f}",
    )
    print(
        "Required contribution margin (%):",
        f"{minimum_margin_pct:.4f}",
    )
    print(
        "Margin shortfall (percentage points):",
        f"{minimum_margin_pct - maximum_margin_pct:.4f}",
    )
    print(
        "Maximum household bill increase (%):",
        f"{diagnostic_bill_increase_pct.max():.4f}",
    )
    print("Diagnostic result only; no tariff has been selected.")
else:
    print(margin_limit_result.message)

,diagnostic_rate_gbp_per_kwh
Peak,0.180000
Shoulder,0.129065
Off-peak,0.100000


Maximum achievable contribution margin (%): 24.0289
Required contribution margin (%): 25.0000
Margin shortfall (percentage points): 0.9711
Maximum household bill increase (%): 2.0000
Diagnostic result only; no tariff has been selected.


In [8]:
import json

feasibility_record = {
    "analysis_period": "2013-01-01 to 2013-06-30 (UTC interval starts)",
    "households": int(len(design_band_kwh)),
    "scenario": "All selected households switch to one candidate tariff",
    "grid_combinations_evaluated": int(len(tariff_search)),
    "grid_feasible_combinations": int(len(feasible_tariffs)),
    "continuous_optimisation_status": "infeasible",
    "rate_bounds_gbp_per_kwh": {
        band: [float(lower), float(upper)]
        for band, (lower, upper) in zip(bands, rate_bounds)
    },
    "required_contribution_margin_pct": float(minimum_margin_pct),
    "maximum_household_bill_increase_pct": float(
        maximum_bill_increase_pct
    ),
    "maximum_achievable_margin_pct_under_bill_limit": float(
        maximum_margin_pct
    ),
    "margin_shortfall_percentage_points": float(
        minimum_margin_pct - maximum_margin_pct
    ),
    "diagnostic_rates_gbp_per_kwh": diagnostic_rates.to_dict(),
    "decision": "Do not select an all-household tariff under these assumptions",
    "limitations": [
        "Variable costs are illustrative assumptions.",
        "Consumption is held unchanged across tariffs.",
        "Standing charges and fixed costs are excluded.",
        "The conclusion applies only within the specified rate bounds.",
        "Validation-period performance has not been evaluated."
    ],
}

record_file = (
    processed_folder / "design_tariff_feasibility_2013.json"
)

record_file.write_text(
    json.dumps(feasibility_record, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Feasibility record saved: {record_file}")
print("No tariff selected.")
print("Validation-period data has not been used for pricing.")

Feasibility record saved: D:\electricity-tariff-optimisation\data\processed\design_tariff_feasibility_2013.json
No tariff selected.
Validation-period data has not been used for pricing.


In [9]:
optional_switch_results = []

for peak_rate, shoulder_rate, off_peak_rate in price_combinations:
    rates = pd.Series({
        "Peak": peak_rate,
        "Shoulder": shoulder_rate,
        "Off-peak": off_peak_rate,
    })

    candidate_charges = (
        design_band_kwh
        .mul(rates, axis="columns")
        .sum(axis=1)
    )

    switch_mask = candidate_charges < baseline_charges - 1e-9

    final_charges = baseline_charges.where(
        ~switch_mask,
        candidate_charges,
    )

    revenue = final_charges.sum()
    margin_pct = (
        (revenue - total_variable_cost) / revenue * 100
    )

    optional_switch_results.append({
        "peak_rate": peak_rate,
        "shoulder_rate": shoulder_rate,
        "off_peak_rate": off_peak_rate,
        "switching_households": int(switch_mask.sum()),
        "remaining_flat_households": int((~switch_mask).sum()),
        "revenue_gbp": revenue,
        "total_customer_saving_gbp": (
            total_baseline_charge - revenue
        ),
        "contribution_margin_pct": margin_pct,
        "feasible": bool(
            margin_pct >= minimum_margin_pct - 1e-9
            and switch_mask.any()
        ),
    })

optional_switch_search = pd.DataFrame(optional_switch_results)

optional_switch_feasible = (
    optional_switch_search.loc[optional_switch_search["feasible"]]
    .sort_values("total_customer_saving_gbp", ascending=False)
    .reset_index(drop=True)
)

optional_switch_search.to_csv(
    processed_folder / "design_optional_switch_grid_search_2013.csv",
    index=False,
)

print("Scenario: switch only when design-period energy charges are lower.")
print("Portfolio margin includes switching and non-switching households.")
print(f"Combinations evaluated: {len(optional_switch_search)}")
print(f"Feasible combinations: {len(optional_switch_feasible)}")

if optional_switch_feasible.empty:
    print("No tested combination meets the scenario requirements.")
else:
    display(optional_switch_feasible.head(5).round(4))

print("Validation-period data has not been used.")

Scenario: switch only when design-period energy charges are lower.
Portfolio margin includes switching and non-switching households.
Combinations evaluated: 48
Feasible combinations: 28


,peak_rate,shoulder_rate,off_peak_rate,switching_households,remaining_flat_households,revenue_gbp,total_customer_saving_gbp,contribution_margin_pct,feasible
0,0.22,0.13,0.08,220,28,73260.3916,4879.6384,25.3722,True
1,0.18,0.14,0.08,224,24,73382.9640,4757.0660,25.4968,True
2,0.24,0.12,0.10,221,27,73799.8113,4340.2187,25.9176,True
3,0.22,0.14,0.06,154,94,73846.0688,4293.9612,25.9641,True
4,0.20,0.13,0.10,230,18,73938.3497,4201.6803,26.0565,True


Validation-period data has not been used.


In [10]:
assert not optional_switch_feasible.empty

selected_option = optional_switch_feasible.iloc[0]

locked_rates = pd.Series({
    "Peak": float(selected_option["peak_rate"]),
    "Shoulder": float(selected_option["shoulder_rate"]),
    "Off-peak": float(selected_option["off_peak_rate"]),
})

design_candidate_charges = (
    design_band_kwh
    .mul(locked_rates, axis="columns")
    .sum(axis=1)
)

locked_assignments = pd.DataFrame(
    index=design_band_kwh.index
)

locked_assignments["assigned_tariff"] = "Flat"
locked_assignments.loc[
    design_candidate_charges < baseline_charges - 1e-9,
    "assigned_tariff",
] = "Time-of-use"

locked_assignments.to_csv(
    processed_folder / "locked_customer_tariff_assignments_2013.csv"
)

locked_config = {
    "scenario": "Design-based selective switching",
    "selection_period": "2013-01-01 to 2013-06-30 (UTC interval starts)",
    "evaluation_period": "2013-07-01 to 2013-12-31 (UTC interval starts)",
    "selection_objective": (
        "Maximise total design-period customer savings "
        "among feasible grid combinations"
    ),
    "rates_gbp_per_kwh": locked_rates.to_dict(),
    "baseline_rate_gbp_per_kwh": float(baseline_rate),
    "variable_cost_rates_gbp_per_kwh": variable_cost_rates.to_dict(),
    "minimum_portfolio_margin_pct": float(minimum_margin_pct),
    "validation_bill_increase_limit_pct": float(
        maximum_bill_increase_pct
    ),
    "assignment_rule": (
        "Assign time-of-use only when design-period charges "
        "are lower; otherwise retain flat pricing"
    ),
    "validation_rule": (
        "Keep prices and assignments unchanged during evaluation"
    ),
    "limitations": [
        "Assignments simulate switching, not actual customer uptake.",
        "Consumption is held unchanged across tariffs.",
        "Standing charges and fixed costs are excluded.",
        "Design-period savings do not guarantee validation-period savings."
    ],
}

config_file = processed_folder / "locked_optional_switch_tariff_2013.json"

config_file.write_text(
    json.dumps(locked_config, indent=2) + "\n",
    encoding="utf-8",
)

display(locked_rates.rename("GBP per kWh").to_frame())
print(locked_assignments["assigned_tariff"].value_counts())
print("Tariff rates and customer assignments saved.")
print("Validation-period data has not been used for selection.")

,GBP per kWh
Peak,0.22
Shoulder,0.13
Off-peak,0.08


assigned_tariff
Time-of-use    220
Flat            28
Name: count, dtype: int64
Tariff rates and customer assignments saved.
Validation-period data has not been used for selection.


In [11]:
import duckdb

database_file = processed_folder / "electricity_analysis.duckdb"

with duckdb.connect(
    str(database_file),
    read_only=True,
) as validation_connection:
    validation_usage = validation_connection.execute("""
        SELECT
            household_id,
            interval_start_utc,
            energy_kwh
        FROM analysis_consumption_2013
        WHERE interval_start_utc >= TIMESTAMP '2013-07-01'
          AND interval_start_utc < TIMESTAMP '2014-01-01'
    """).df()

local_time = pd.to_datetime(
    validation_usage["interval_start_utc"],
    utc=True,
).dt.tz_convert("Europe/London")

validation_usage["candidate_band"] = "Shoulder"

validation_usage.loc[
    local_time.dt.hour < 7,
    "candidate_band",
] = "Off-peak"

validation_usage.loc[
    (local_time.dt.dayofweek < 5)
    & (local_time.dt.hour >= 16)
    & (local_time.dt.hour < 20),
    "candidate_band",
] = "Peak"

validation_band_kwh = (
    validation_usage
    .groupby(["household_id", "candidate_band"])["energy_kwh"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=bands, fill_value=0)
)

assert len(validation_usage) == 248 * 184 * 48
assert validation_usage["household_id"].nunique() == 248
assert not validation_usage.duplicated(
    ["household_id", "interval_start_utc"]
).any()
assert set(validation_band_kwh.index) == set(locked_assignments.index)
assert validation_usage["energy_kwh"].notna().all()
assert (validation_usage["energy_kwh"] >= 0).all()

validation_band_kwh.to_csv(
    processed_folder / "customer_validation_band_kwh_2013.csv"
)

print(f"Validation households: {len(validation_band_kwh)}")
print(f"Validation rows: {len(validation_usage):,}")
print(
    "Validation energy (kWh):",
    f"{validation_band_kwh.sum().sum():,.3f}",
)
print("Pricing bands use UK local interval-start time.")
print("Bank holidays are treated as ordinary weekdays.")
print("Locked prices and customer assignments remain unchanged.")

display(validation_band_kwh.head())

Validation households: 248
Validation rows: 2,190,336
Validation energy (kWh): 489,289.068
Pricing bands use UK local interval-start time.
Bank holidays are treated as ordinary weekdays.
Locked prices and customer assignments remain unchanged.


candidate_band,Peak,Shoulder,Off-peak
household_id,,,
MAC000002,333.208000,1311.276999,382.912000
MAC000040,823.393000,3753.777000,1246.700999
MAC000100,343.128000,1502.217999,245.590000
MAC000145,373.345999,1609.445000,493.308000
MAC000162,88.997000,372.151000,146.777000


In [12]:
validation_impact = locked_assignments.copy()

validation_impact["energy_kwh"] = (
    validation_band_kwh.sum(axis=1)
)

validation_impact["baseline_charge_gbp"] = (
    validation_impact["energy_kwh"] * baseline_rate
)

validation_impact["tou_charge_gbp"] = (
    validation_band_kwh
    .mul(locked_rates, axis="columns")
    .sum(axis=1)
)

is_tou = (
    validation_impact["assigned_tariff"] == "Time-of-use"
)

validation_impact["assigned_charge_gbp"] = (
    validation_impact["baseline_charge_gbp"].where(
        ~is_tou,
        validation_impact["tou_charge_gbp"],
    )
)

validation_impact["saving_gbp"] = (
    validation_impact["baseline_charge_gbp"]
    - validation_impact["assigned_charge_gbp"]
)

validation_impact["saving_pct"] = (
    validation_impact["saving_gbp"]
    / validation_impact["baseline_charge_gbp"]
    * 100
)

validation_impact["variable_cost_gbp"] = (
    validation_band_kwh
    .mul(variable_cost_rates, axis="columns")
    .sum(axis=1)
)

validation_revenue = (
    validation_impact["assigned_charge_gbp"].sum()
)

validation_cost = (
    validation_impact["variable_cost_gbp"].sum()
)

validation_margin_pct = (
    (validation_revenue - validation_cost)
    / validation_revenue * 100
)

bill_increase_pct = -validation_impact["saving_pct"]

validation_summary = pd.Series({
    "households": len(validation_impact),
    "households_assigned_tou": int(is_tou.sum()),
    "households_remaining_flat": int((~is_tou).sum()),
    "baseline_charge_gbp": (
        validation_impact["baseline_charge_gbp"].sum()
    ),
    "assigned_charge_gbp": validation_revenue,
    "total_customer_saving_gbp": (
        validation_impact["saving_gbp"].sum()
    ),
    "households_paying_less": int(
        (validation_impact["saving_gbp"] > 1e-9).sum()
    ),
    "households_paying_more": int(
        (validation_impact["saving_gbp"] < -1e-9).sum()
    ),
    "max_bill_increase_pct": bill_increase_pct.max(),
    "households_above_2pct_increase": int(
        (bill_increase_pct > maximum_bill_increase_pct + 1e-9).sum()
    ),
    "contribution_margin_pct": validation_margin_pct,
})

validation_impact.to_csv(
    processed_folder / "validation_locked_tariff_customer_impact_2013.csv"
)

validation_summary.to_frame("value").to_csv(
    processed_folder / "validation_locked_tariff_summary_2013.csv"
)

display(validation_summary.to_frame("value").round(4))

print(
    "Margin constraint passed:",
    bool(validation_margin_pct >= minimum_margin_pct - 1e-9),
)
print(
    "Household bill constraint passed:",
    bool(
        bill_increase_pct.max()
        <= maximum_bill_increase_pct + 1e-9
    ),
)
print("Evaluation completed without changing prices or assignments.")

,value
households,248.0000
households_assigned_tou,220.0000
households_remaining_flat,28.0000
baseline_charge_gbp,69616.0486
assigned_charge_gbp,65690.5502
total_customer_saving_gbp,3925.4984
households_paying_less,213.0000
households_paying_more,7.0000
max_bill_increase_pct,2.7651
households_above_2pct_increase,1.0000


Margin constraint passed: True
Household bill constraint passed: False
Evaluation completed without changing prices or assignments.


In [13]:
paying_more_review = validation_impact.loc[
    validation_impact["saving_gbp"] < -1e-9,
    [
        "assigned_tariff",
        "baseline_charge_gbp",
        "assigned_charge_gbp",
        "saving_gbp",
        "saving_pct",
    ],
].copy()

paying_more_review["bill_increase_pct"] = (
    -paying_more_review["saving_pct"]
)

design_shares = design_band_kwh.div(
    design_band_kwh.sum(axis=1),
    axis=0,
) * 100

validation_shares = validation_band_kwh.div(
    validation_band_kwh.sum(axis=1),
    axis=0,
) * 100

for band, label in [
    ("Peak", "peak"),
    ("Off-peak", "off_peak"),
]:
    paying_more_review[f"design_{label}_share_pct"] = (
        design_shares[band]
    )
    paying_more_review[f"validation_{label}_share_pct"] = (
        validation_shares[band]
    )
    paying_more_review[f"{label}_share_change_pp"] = (
        validation_shares[band] - design_shares[band]
    )

paying_more_review["above_increase_limit"] = (
    paying_more_review["bill_increase_pct"]
    > maximum_bill_increase_pct + 1e-9
)

paying_more_review = paying_more_review.sort_values(
    "bill_increase_pct",
    ascending=False,
)

paying_more_review.to_csv(
    processed_folder / "validation_paying_more_review_2013.csv"
)

display(paying_more_review.round(4))

print("Share changes are measured in percentage points.")
print("Prices and customer assignments remain unchanged.")

,assigned_tariff,baseline_charge_gbp,assigned_charge_gbp,saving_gbp,saving_pct,bill_increase_pct,design_peak_share_pct,validation_peak_share_pct,peak_share_change_pp,design_off_peak_share_pct,validation_off_peak_share_pct,off_peak_share_change_pp,above_increase_limit
household_id,,,,,,,,,,,,,
MAC004646,Time-of-use,494.7528,508.4333,-13.6805,-2.7651,2.7651,20.7362,25.7801,5.0439,17.8023,13.9757,-3.8266,True
MAC002806,Time-of-use,168.7549,171.9467,-3.1918,-1.8914,1.8914,19.7668,21.7349,1.9682,15.5499,9.1809,-6.3691,False
MAC004908,Time-of-use,702.4635,710.7133,-8.2497,-1.1744,1.1744,15.9363,20.0214,4.0851,5.2084,8.1367,2.9283,False
MAC002170,Time-of-use,395.6797,399.4472,-3.7675,-0.9522,0.9522,18.8312,20.7768,1.9456,9.4809,10.1288,0.6479,False
MAC002517,Time-of-use,280.4749,281.1121,-0.6372,-0.2272,0.2272,18.4534,20.6709,2.2175,13.5119,12.0011,-1.5108,False
MAC000390,Time-of-use,311.7674,312.3268,-0.5594,-0.1794,0.1794,20.7749,22.3973,1.6225,14.3196,15.2446,0.9250,False
MAC004532,Time-of-use,334.4112,334.7541,-0.3429,-0.1025,0.1025,18.8331,19.2914,0.4583,9.6190,9.8726,0.2537,False


Share changes are measured in percentage points.
Prices and customer assignments remain unchanged.


In [14]:
protected_impact = validation_impact.copy()

protected_impact["charge_cap_gbp"] = (
    protected_impact["baseline_charge_gbp"]
    * (1 + maximum_bill_increase_pct / 100)
)

protected_impact["rebate_gbp"] = (
    protected_impact["assigned_charge_gbp"]
    - protected_impact["charge_cap_gbp"]
).clip(lower=0)

protected_impact["protected_charge_gbp"] = (
    protected_impact["assigned_charge_gbp"]
    - protected_impact["rebate_gbp"]
)

protected_revenue = (
    protected_impact["protected_charge_gbp"].sum()
)

protected_margin_pct = (
    (protected_revenue - validation_cost)
    / protected_revenue * 100
)

protected_increase_pct = (
    protected_impact["protected_charge_gbp"]
    / protected_impact["baseline_charge_gbp"] - 1
) * 100

protection_summary = pd.Series({
    "households_receiving_rebate": int(
        (protected_impact["rebate_gbp"] > 1e-9).sum()
    ),
    "total_rebate_gbp": protected_impact["rebate_gbp"].sum(),
    "protected_revenue_gbp": protected_revenue,
    "total_customer_saving_gbp": (
        protected_impact["baseline_charge_gbp"].sum()
        - protected_revenue
    ),
    "contribution_margin_pct": protected_margin_pct,
    "max_bill_increase_pct": protected_increase_pct.max(),
})

protected_impact.to_csv(
    processed_folder / "validation_bill_protection_scenario_2013.csv"
)

protection_summary.to_frame("value").to_csv(
    processed_folder / "validation_bill_protection_summary_2013.csv"
)

display(protection_summary.to_frame("value").round(4))

print(
    "Margin constraint passed:",
    bool(protected_margin_pct >= minimum_margin_pct - 1e-9),
)
print(
    "Household bill constraint passed:",
    bool(
        protected_increase_pct.max()
        <= maximum_bill_increase_pct + 1e-9
    ),
)
print("Post-validation mitigation scenario; not an independent test.")
print("Protection applies to total half-year energy charges.")
print("Original validation results remain unchanged.")

,value
households_receiving_rebate,1.0000
total_rebate_gbp,3.7855
protected_revenue_gbp,65686.7647
total_customer_saving_gbp,3929.2839
contribution_margin_pct,25.4808
max_bill_increase_pct,2.0000


Margin constraint passed: True
Household bill constraint passed: True
Post-validation mitigation scenario; not an independent test.
Protection applies to total half-year energy charges.
Original validation results remain unchanged.


In [15]:
cost_stress_results = []

for cost_increase_pct in [0, 5, 10, 20]:
    stressed_cost = (
        validation_cost * (1 + cost_increase_pct / 100)
    )

    contribution = protected_revenue - stressed_cost
    margin_pct = contribution / protected_revenue * 100

    cost_stress_results.append({
        "cost_increase_pct": cost_increase_pct,
        "revenue_after_rebate_gbp": protected_revenue,
        "variable_cost_gbp": stressed_cost,
        "contribution_gbp": contribution,
        "contribution_margin_pct": margin_pct,
        "margin_constraint_pass": bool(
            margin_pct >= minimum_margin_pct - 1e-9
        ),
    })

cost_stress_table = pd.DataFrame(cost_stress_results)

cost_headroom_pct = (
    protected_revenue
    * (1 - minimum_margin_pct / 100)
    / validation_cost - 1
) * 100

cost_stress_table.to_csv(
    processed_folder / "validation_protected_tariff_cost_stress_2013.csv",
    index=False,
)

display(cost_stress_table.round(4))

print(
    "Maximum variable-cost increase before falling below "
    f"the margin target (%): {cost_headroom_pct:.4f}"
)
print("Prices, consumption and rebates are held unchanged.")
print("Cost increases are illustrative stress scenarios.")

,cost_increase_pct,revenue_after_rebate_gbp,variable_cost_gbp,contribution_gbp,contribution_margin_pct,margin_constraint_pass
0,0,65686.7647,48949.2461,16737.5186,25.4808,True
1,5,65686.7647,51396.7084,14290.0563,21.7548,False
2,10,65686.7647,53844.1707,11842.5940,18.0289,False
3,20,65686.7647,58739.0953,6947.6694,10.5770,False


Maximum variable-cost increase before falling below the margin target (%): 0.6452
Prices, consumption and rebates are held unchanged.
Cost increases are illustrative stress scenarios.


In [16]:
quality_flags = pd.read_csv(
    project_folder
    / "reports"
    / "data_quality"
    / "selected_customer_review_flags_2013.csv",
    index_col="household_id",
)

review_flags = (
    quality_flags["review_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert review_flags.isin(["true", "false"]).all()

review_flags = review_flags.map({
    "true": True,
    "false": False,
}).reindex(protected_impact.index)

assert review_flags.notna().all()

quality_sensitivity_results = []

for sample_name, sample_mask in [
    ("All households", review_flags.notna()),
    ("Excluding flagged households", ~review_flags),
]:
    sample = protected_impact.loc[sample_mask]

    revenue = sample["protected_charge_gbp"].sum()
    variable_cost = sample["variable_cost_gbp"].sum()

    bill_increase = (
        sample["protected_charge_gbp"]
        / sample["baseline_charge_gbp"] - 1
    ) * 100

    quality_sensitivity_results.append({
        "sample": sample_name,
        "households": len(sample),
        "customer_saving_gbp": (
            sample["baseline_charge_gbp"].sum() - revenue
        ),
        "customer_saving_pct": (
            (sample["baseline_charge_gbp"].sum() - revenue)
            / sample["baseline_charge_gbp"].sum() * 100
        ),
        "contribution_margin_pct": (
            (revenue - variable_cost) / revenue * 100
        ),
        "max_bill_increase_pct": bill_increase.max(),
        "rebate_gbp": sample["rebate_gbp"].sum(),
    })

quality_sensitivity = pd.DataFrame(
    quality_sensitivity_results
)

quality_sensitivity.to_csv(
    processed_folder / "validation_quality_sensitivity_2013.csv",
    index=False,
)

display(quality_sensitivity.round(4))

print("Original prices and customer assignments retained.")
print("Review flags indicate unusual patterns, not confirmed errors.")
print("Full-year flags make this a retrospective sensitivity check.")

,sample,households,customer_saving_gbp,customer_saving_pct,contribution_margin_pct,max_bill_increase_pct,rebate_gbp
0,All households,248,3929.2839,5.6442,25.4808,2.0,3.7855
1,Excluding flagged households,244,3817.3664,5.5330,25.4856,2.0,3.7855


Original prices and customer assignments retained.
Review flags indicate unusual patterns, not confirmed errors.
Full-year flags make this a retrospective sensitivity check.


## Findings and decision implications

### All-household switching

No tariff within the specified rate bounds met both the 25% portfolio
contribution margin requirement and the 2% maximum household energy
charge increase. This was checked using grid search and linear programming.

### Selective switching

The design-period grid search selected rates of GBP 0.22, 0.13 and 0.08
per kWh for peak, shoulder and off-peak consumption respectively.
Assignments were fixed before evaluation: 220 households switched and
28 retained flat pricing.

In the validation period, the locked scenario produced aggregate energy
charge savings of GBP 3,925.50 and a contribution margin of 25.49%.
However, seven households paid more than under flat pricing, including
one whose increase exceeded 2%.

### Post-validation bill protection

An illustrative half-year rebate of GBP 3.79 limited the maximum
household increase to 2%. Aggregate savings became GBP 3,929.28,
with a contribution margin of 25.48%.

This mitigation was proposed after inspecting validation results.
It is not an independently validated protection strategy or a monthly
bill guarantee.

### Sensitivity and recommendation

Excluding four households flagged for unusual zero-reading patterns
changed aggregate savings from 5.64% to 5.53% and the contribution
margin from 25.48% to 25.49%. Prices and assignments were unchanged.

The protected scenario had limited cost headroom: an increase of
approximately 0.65% in assumed variable costs would reduce the
contribution margin below 25%.

The findings support further investigation of selective switching,
rather than immediate rollout. Priorities include validating supplier
cost assumptions, evaluating additional periods, and assessing customer
protection and monitoring requirements.

### Interpretation limits

- Costs and candidate tariffs are illustrative.
- Consumption is held unchanged; customer uptake is simulated.
- Standing charges, fixed costs and implementation costs are excluded.
- Contribution is not net profit.
- Full-year completeness informed sample selection.
- Quality flags use full-year data and support retrospective sensitivity only.
- Results concern this historical UK sample, not current New Zealand tariffs.

In [18]:
segment_validation = validation_impact.join(
    customer_features[["usage_segment"]],
    how="left",
    validate="one_to_one",
)

assert segment_validation["usage_segment"].notna().all()

segment_validation["assigned_tou"] = (
    segment_validation["assigned_tariff"] == "Time-of-use"
)

segment_validation["paying_more"] = (
    segment_validation["saving_gbp"] < -1e-9
)

segment_validation["above_increase_limit"] = (
    -segment_validation["saving_pct"]
    > maximum_bill_increase_pct + 1e-9
)

segment_validation_summary = (
    segment_validation
    .groupby("usage_segment")
    .agg(
        households=("assigned_tariff", "size"),
        assigned_tou=("assigned_tou", "sum"),
        baseline_charge_gbp=("baseline_charge_gbp", "sum"),
        total_saving_gbp=("saving_gbp", "sum"),
        median_saving_pct=("saving_pct", "median"),
        households_paying_more=("paying_more", "sum"),
        households_above_limit=("above_increase_limit", "sum"),
    )
)

segment_validation_summary["aggregate_saving_pct"] = (
    segment_validation_summary["total_saving_gbp"]
    / segment_validation_summary["baseline_charge_gbp"]
    * 100
)

segment_validation_summary.to_csv(
    processed_folder / "validation_segment_outcomes_2013.csv"
)

display(segment_validation_summary.round(4))

print("Segments were defined using design-period consumption.")
print("Results reflect locked tariff assignments before rebates.")
print("Flat-rate households are included with zero savings.")
print("No customer segments or assignments were changed.")

,households,assigned_tou,baseline_charge_gbp,total_saving_gbp,median_saving_pct,households_paying_more,households_above_limit,aggregate_saving_pct
usage_segment,,,,,,,,
Higher off-peak share,62,62,20299.1491,2304.8495,8.8197,0,0,11.3544
Higher peak share,61,35,14187.5552,139.3670,0.0000,5,1,0.9823
Other profiles,125,123,35129.3443,1481.2820,4.6764,2,0,4.2167


Segments were defined using design-period consumption.
Results reflect locked tariff assignments before rebates.
Flat-rate households are included with zero savings.
No customer segments or assignments were changed.


In [19]:
assert segment_validation_summary["households"].sum() == 248
assert segment_validation_summary["assigned_tou"].sum() == 220

assert np.isclose(
    segment_validation_summary["total_saving_gbp"].sum(),
    validation_impact["saving_gbp"].sum(),
)

assert (
    segment_validation_summary["households_paying_more"].sum()
    == 7
)

assert (
    segment_validation_summary["households_above_limit"].sum()
    == 1
)

customer_results = segment_validation.join(
    protected_impact[
        ["rebate_gbp", "protected_charge_gbp"]
    ],
    validate="one_to_one",
)

customer_results["protected_saving_gbp"] = (
    customer_results["baseline_charge_gbp"]
    - customer_results["protected_charge_gbp"]
)

customer_results["protected_saving_pct"] = (
    customer_results["protected_saving_gbp"]
    / customer_results["baseline_charge_gbp"]
    * 100
)

customer_results["review_flag"] = review_flags.reindex(
    customer_results.index
)

assert customer_results["review_flag"].notna().all()

customer_results.to_csv(
    processed_folder / "customer_tariff_results_2013.csv"
)

print("Segment-to-portfolio reconciliation: PASS")
print(f"Customer result rows: {len(customer_results)}")
print("Results include original outcomes and the rebate scenario.")
print("Customer results saved.")

Segment-to-portfolio reconciliation: PASS
Customer result rows: 248
Results include original outcomes and the rebate scenario.
Customer results saved.
